<a href="https://colab.research.google.com/github/EkaterinaLavlinskaya/ai-agents-for-local-use/blob/main/AI_agents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!apt-get update -qq && apt-get install -y zstd

In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import subprocess
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

In [ ]:
!ollama pull qwen2.5:1.5b

In [ ]:
%pip install -U langchain langchain-core langchain-ollama langgraph

In [ ]:
with open("Modelfile", "w") as f:
    f.write("FROM qwen2.5:7b\n")

In [ ]:
!ollama create my-qwen -f Modelfile

In [ ]:
import subprocess
import time

# Запускаем сервер Ollama в фоне
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

# Даём серверу время запуститься
time.sleep(3)

# Проверяем, что сервер отвечает (опционально)
!curl http://localhost:11434/api/tags

In [ ]:
from datetime import datetime

from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_ollama import ChatOllama
from langgraph.checkpoint.memory import InMemorySaver

CHAT_MODEL = "qwen2.5:1.5b"   # Чат-модель Ollama. Должна поддерживать вызов инструментов.

SYSTEM_PROMPT = (
    "You are a helpful assistant with access to tools for getting the current time and counting words in text. "
    "Use tools when the user's request needs one. "
    "If the question doesn't need a tool, answer directly. "
    "If a tool returns an error, explain the error plainly."
)

# ----- Tools -----
@tool
def current_time() -> str:
    """Возвращает текущую локальную дату и время.
    Используйте этот инструмент, когда пользователь спрашивает, который час или день.
    """
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")

@tool
def word_count(text: str) -> int:
    """Подсчитывает количество слов в тексте.
    Используйте этот метод, когда пользователь спрашивает, сколько слов в тексте,
    или просит вас подсчитать количество слов в чем-то, чем он поделился.
    Возвращает количество слов в виде целого числа.
    """
    return len(text.split())

TOOLS = [current_time, word_count]

# ----- Agent -----
def build_agent():
    model = ChatOllama(model="my-qwen", temperature=0.7, repeat_penalty=1.1)

    # InMemorySaver хранит историю диалога в памяти, индексируя ее по идентификатору потока.
    # После завершения процесса история удаляется из-за кратковременной памяти.
    checkpointer = InMemorySaver()

    return create_agent(
        model=model,
        tools=TOOLS,
        system_prompt=SYSTEM_PROMPT,
        checkpointer=checkpointer,
    )

def main():
    agent = build_agent()

    # Идентификатор потока сообщает чекпойнтеру, какой диалог нужно загрузить и сохранить.
    config = {"configurable": {"thread_id": "thread"}}

    print("Ready! Ask the agent something. It remembers the conversation.\n")

    # Отследим, сколько сообщений было до этого хода, чтобы выделить
    # только новые (вызовы инструментов + окончательный ответ) из возвращенного состояния.
    prev_message_count = 0

    while True:
        question = input("You: ").strip()
        if not question or question.lower() == "exit":
            break

        result = agent.invoke(
            {"messages": [{"role": "user", "content": question}]},
            config=config,
        )

        # Смотрим только на сообщения, добавленные в этот ход, а не на всю историю.
        new_messages = result["messages"][prev_message_count:]

        # Выводим все вызовы инструментов, сделанные в этот ход.
        for msg in new_messages:
            tool_calls = getattr(msg, "tool_calls", None)
            if tool_calls:
                for call in tool_calls:
                    print(f"[tool call] {call['name']}({call['args']})")

        print(f"\nAnswer: {result['messages'][-1].content}\n")

        # Обновляем счетчик для следующего хода.
        prev_message_count = len(result["messages"])

if __name__ == "__main__":
    main()